# After the Water: Pakistan Floods 2022
## Notebook A · Session 2 — Folders inside folders

**How it works.** Work in pairs and swap the typist at every task. Run the cells from the top (Shift+Enter). Where it says `TODO`, write your code, run the cell, then run the **test cell** under it and aim for all `[PASS]`. Type your answers in the *My answer* cells.

Keep the **Session 2 Helper Sheet** open. You need `s1_classified_rows.csv` in this folder: your own from Session 1, or the ready-made copy in the zip.

| Task | Function | What it does |
| --- | --- | --- |
| 1 | `build_tree(rows)` | flat rows → a tree |
| 2 | `count_districts(place)` | your first recursion |
| 3 | `total_population(place)` | the same pattern again |
| 4 | `find_path(place, name)` | bonus, if you finish early |

## Where we left off

In Session 1 we found that the flood file stores its structure by position: a province row, then its districts, then the next province. We also found two rows called "Islamabad": the territory, and the one district inside it.

Today we turn the flat rows into a tree, folders inside folders:

```
Pakistan
 |-- Azad Kashmir
 |     |-- Bagh
 |     |-- ...
 |-- ...
 |-- Sindh
       |-- Dadu
       |-- ...
```

Then we use recursion to ask: do the numbers add up, and which districts were hit hardest?

**Today's plan** (75 minutes)

```
 0-10 min   Sections 1-2  Load your Session 1 file; from text to numbers
10-30 min   Section 3     Task 1: build_tree
30-40 min   Section 4     Task 2: count_districts, your first recursion
40-48 min   Section 4     Task 3: total_population (same pattern)
48-58 min   Section 5     Run the audit: do the numbers add up?
58-70 min   Section 6     Which districts were hit hardest? (+ a mystery)
70-75 min   Section 7     Save your work for the lab
Finished early?           Task 4: find_path
```

In [1]:
import csv
from helpers import banner, check, summary

banner("READY")
print("banner() is loaded. Let's load the Session 1 file.")


########################################################################
#  READY
########################################################################
banner() is loaded. Let's load the Session 1 file.


## Section 1 — Load your Session 1 file

This time we use csv.DictReader. It gives each row as a dictionary, so you reach a value by its column name instead of its position:

```
With csv.reader:      row[1]
With csv.DictReader:  row["name"]
```

In [2]:
banner("SECTION 1 — Loading s1_classified_rows.csv")

FILE = "s1_classified_rows.csv"

with open(FILE, "r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

print("Rows loaded:", len(rows), "  (expected 168)")

print()
print("The first row, as a dictionary:")
print(rows[0])
print()
print("Its name: ", rows[0]["name"])
print("Its level:", rows[0]["level"])


########################################################################
#  SECTION 1 — Loading s1_classified_rows.csv
########################################################################
Rows loaded: 168   (expected 168)

The first row, as a dictionary:
{'level': 'country', 'name': 'Pakistan', 'area_km2': '877964.6739099998', 'population': '231088952.39900398', 'flood_km2 01-31 Aug': '83557.111748835', 'flood_km2 25-31 Aug': '68963.77902800002', 'flood_km2 01-07 Sep': '59789.41879099998', 'flood_km2 08-14 Sep': '53172.069765629014', 'flood_km2 15-21 Sep': '45579.941169', 'flood_km2 26 Sep-02 Oct': '41737.804367', 'flood_km2 03-09 Oct': '36596.468339', 'flood_km2 11-17 Oct': '33631.48513', 'flood_km2 14-20 Nov': '24660.371607999998', 'exposed 01-31 Aug': '31251773.832889862', 'exposed 25-31 Aug': '23717896.32008001', 'exposed 01-07 Sep': '21814554.30890399', 'exposed 08-14 Sep': '19035055.904096525', 'exposed 15-21 Sep': '16672766.843905997', 'exposed 26 Sep-02 Oct': '14851871.632

Every value is still a string. float("2597.07") turns text into a number.

## Section 2 — From text to numbers

**Given code** — window_values(row, prefix)

Each row has 9 flood columns ("flood_km2 01-31 Aug", ...) and 9 exposed columns ("exposed 01-31 Aug", ...). window_values(row, prefix) gives back the 9 values for one prefix, as a list of numbers.

In [3]:
banner("GIVEN — window_values")

WINDOW_LABELS = ["01-31 Aug", "25-31 Aug", "01-07 Sep", "08-14 Sep", "15-21 Sep",
                 "26 Sep-02 Oct", "03-09 Oct", "11-17 Oct", "14-20 Nov"]


def window_values(row, prefix):
    """Return the 9 values for this prefix ("flood_km2 " or "exposed ") as floats."""
    values = []
    for label in WINDOW_LABELS:
        values.append(float(row[prefix + label]))
    return values


dadu_row = None
for row in rows:
    if row["name"] == "Dadu":
        dadu_row = row

print("Dadu's row, people exposed (text):  ", dadu_row["exposed 01-31 Aug"], "...")
print("window_values(dadu_row, 'exposed '): ", window_values(dadu_row, "exposed "))
print("window_values(dadu_row, 'flood_km2 '):", window_values(dadu_row, "flood_km2 "))


########################################################################
#  GIVEN — window_values
########################################################################
Dadu's row, people exposed (text):   1262147.07166 ...
window_values(dadu_row, 'exposed '):  [1262147.07166, 1239852.658523, 1213532.2721230001, 1127861.4092020001, 1060532.025413, 917451.246878, 819198.37995, 761368.7506230001, 596621.718724]
window_values(dadu_row, 'flood_km2 '): [2627.515, 2597.0730000000003, 2503.227, 2408.9040000000005, 2303.034, 2005.3819999999998, 1809.7700000000002, 1700.107, 1402.125]


## Given code — make_place(row)

make_place turns a row into a place: a dictionary with the place's name, its numbers, and a "children" list for the places inside it. Pakistan's children will be its provinces, and a province's children its districts. A district's children list stays empty.

A row comes from the file (all strings). A place is what we build. Your functions today all work on places.

In [4]:
banner("GIVEN — make_place: Dadu as a place")

def make_place(row):
    """Turn one row into a place: a dictionary with an empty children list."""
    return {
        "name": row["name"],
        "level": row["level"],
        "population": float(row["population"]),
        "flood": window_values(row, "flood_km2 "),      # 9 numbers, km2
        "exposed": window_values(row, "exposed "),      # 9 numbers, people
        "children": [],
    }


example = make_place(dadu_row)
print("Dadu as a place:")
print("  name:      ", example["name"])
print("  level:     ", example["level"])
print("  population:", example["population"])
print("  exposed:   ", example["exposed"])
print("  children:  ", example["children"])


########################################################################
#  GIVEN — make_place: Dadu as a place
########################################################################
Dadu as a place:
  name:       Dadu
  level:      district
  population: 1846783.7820489998
  exposed:    [1262147.07166, 1239852.658523, 1213532.2721230001, 1127861.4092020001, 1060532.025413, 917451.246878, 819198.37995, 761368.7506230001, 596621.718724]
  children:   []


## Section 3 — Turn the rows into a tree

### Task 1 — build_tree(rows)

**What to do**

Go through the rows in order. Turn each row into a place with make_place(row), then:

- a "country" row  -> this is Pakistan. Keep it in a variable called top.
- a "province" row -> add it to top's children. It becomes the current province.
- a "district" row -> add it to the current province's children.

Return top at the end.

The Islamabad rule: a "province" row with the same name as the current province is really its district. Set its "level" to "district" and add it to the current province's children.

No recursion here: one loop, and two variables (top and current_province).

**What it should return**

```
top["name"]                                  -> "Pakistan"
top["children"]                              -> a list of 7 provinces
top["children"][0]["name"]                   -> "Azad Kashmir"
top["children"][0]["children"][0]["name"]    -> "Bagh"
```

In [5]:
def build_tree(rows):
    """Turn the flat rows into a tree. Return the top place (Pakistan)."""
    # >>> TODO (Task 1): write your code here <<<
    top = None
    current_province = None
    for row in rows:
        current = make_place(row)
        if row["level"] == "country":
            top = current 
        elif row["level"] == "province":
            if current_province is not None and current["name"] == current_province["name"]:
                current["level"] = "district"
                current_province["children"].append(current)
            else:
                top["children"].append(current)
                current_province = current
        elif row["level"] == "district":
            current_province["children"].append(current)
    return top


In [6]:
# TEST — Task 1: build_tree      (given code: don't change this cell)
# Run the cell above first, and again every time you change build_tree.
results = []
test_tree = build_tree(rows)
if test_tree is None:
    results.append(check("build_tree(rows)", test_tree, "the Pakistan place"))
else:
    results.append(check('tree["name"]', test_tree["name"], "Pakistan"))
    results.append(check('len(tree["children"])   (provinces)', len(test_tree["children"]), 7))
    if len(test_tree["children"]) == 7:
        sindh = test_tree["children"][6]
        islamabad = test_tree["children"][3]
        results.append(check('tree["children"][6]["name"]', sindh["name"], "Sindh"))
        results.append(check("districts in Sindh", len(sindh["children"]), 29))
        results.append(check('tree["children"][3]["name"]', islamabad["name"], "Islamabad"))
        results.append(check("districts in Islamabad   (the Islamabad rule!)",
                             len(islamabad["children"]), 1))
        if len(islamabad["children"]) == 1:
            results.append(check("level of Islamabad's district",
                                 islamabad["children"][0]["level"], "district"))
        total = 0
        for province in test_tree["children"]:
            total = total + len(province["children"])
        results.append(check("districts in all 7 provinces", total, 160))
summary("Task 1", results)

  [PASS] tree["name"] gave 'Pakistan'
  [PASS] len(tree["children"])   (provinces) gave 7
  [PASS] tree["children"][6]["name"] gave 'Sindh'
  [PASS] districts in Sindh gave 29
  [PASS] tree["children"][3]["name"] gave 'Islamabad'
  [PASS] districts in Islamabad   (the Islamabad rule!) gave 1
  [PASS] level of Islamabad's district gave 'district'
  [PASS] districts in all 7 provinces gave 160

  Task 1: 8 of 8 tests passed.
  Well done! Move on to the next task.


In [7]:
banner("TASK 1 — Your tree")

tree = build_tree(rows)

if tree is None:
    print("build_tree isn't written yet. Later sections will be skipped until it is.")
else:
    print(tree["name"], "contains", len(tree["children"]), "provinces:")
    for province in tree["children"]:
        print("  ", province["name"], "-", len(province["children"]), "districts")

# GIVEN: the later sections need build_tree to work.
ready = tree is not None


########################################################################
#  TASK 1 — Your tree
########################################################################
Pakistan contains 7 provinces:
   Azad Kashmir - 10 districts
   Balochistan - 35 districts
   Gilgit Baltistan - 14 districts
   Islamabad - 1 districts
   Khyber Pakhtunkhwa - 35 districts
   Punjab - 36 districts
   Sindh - 29 districts


## Section 4 — Recursion

### Task 2 — your first recursion: count_districts(place)

Write count_districts(place): how many districts are inside a place? A district counts as 1.

In [11]:
def count_districts(place):
    """Return how many districts are inside this place (a district counts as 1)."""
    # >>> TODO (Task 2): write your code here <<<
    if place["level"] == "district":
        return 1 
    total = 0
    for i in place["children"]:
        total += count_districts(i)
    return total

In [12]:
# TEST — Task 2: count_districts      (given code: don't change this cell)
# Run the cell above first, and again every time you change count_districts.
results = []
# A tiny made-up tree: Toyland -> North (A, B) and South (C)
def toy_district(name, population):
    return {"name": name, "level": "district", "population": population, "children": []}
north = {"name": "North", "level": "province", "population": 30,
         "children": [toy_district("A", 10), toy_district("B", 20)]}
south = {"name": "South", "level": "province", "population": 30,
         "children": [toy_district("C", 30)]}
toyland = {"name": "Toyland", "level": "country", "population": 60, "children": [north, south]}
print("  On the tiny tree:")
results.append(check("count_districts(district A)", count_districts(north["children"][0]), 1))
results.append(check("count_districts(North)", count_districts(north), 2))
results.append(check("count_districts(Toyland)", count_districts(toyland), 3))
try:
    test_tree = build_tree(rows)
except Exception:
    test_tree = None
print()
if test_tree is None:
    print("  (build_tree isn't working yet, so the real-tree tests were skipped.)")
else:
    print("  On the real tree:")
    results.append(check("count_districts(tree)", count_districts(test_tree), 160))
summary("Task 2", results)

  On the tiny tree:
  [PASS] count_districts(district A) gave 1
  [PASS] count_districts(North) gave 2
  [PASS] count_districts(Toyland) gave 3

  On the real tree:
  [PASS] count_districts(tree) gave 160

  Task 2: 4 of 4 tests passed.
  Well done! Move on to the next task.


In [13]:
banner("TASK 2 — Counting districts")

if ready:
    print("Districts in Islamabad:", count_districts(tree["children"][3]))
    print("Districts in Sindh:    ", count_districts(tree["children"][6]))
    print("Districts in Pakistan: ", count_districts(tree))


########################################################################
#  TASK 2 — Counting districts
########################################################################
Districts in Islamabad: 1
Districts in Sindh:     29
Districts in Pakistan:  160


**Question 1:** Session 1 counted 159 district rows. Why do we get 160 now?

**My answer:** It is counting Islamabad as a district 

## Task 3 — total_population(place)

**What to do**

Write total_population(place): the total population of all the districts inside a place. It has the same shape as count_districts.

In [24]:
def total_population(place):
    """Return the total population of all districts inside this place."""
    # >>> TODO (Task 3): write your code here <<<
    if place["level"] == "district":
        return place["population"] 
    total = 0
    for i in place["children"]:
        total += total_population(i)
    return total

In [25]:
# TEST — Task 3: total_population      (given code: don't change this cell)
# Run the cell above first, and again every time you change total_population.
results = []
# A tiny made-up tree: Toyland -> North (A 10, B 20) and South (C 30)
def toy_district(name, population):
    return {"name": name, "level": "district", "population": population, "children": []}
north = {"name": "North", "level": "province", "population": 30,
         "children": [toy_district("A", 10), toy_district("B", 20)]}
south = {"name": "South", "level": "province", "population": 30,
         "children": [toy_district("C", 30)]}
toyland = {"name": "Toyland", "level": "country", "population": 60, "children": [north, south]}
print("  On the tiny tree:")
results.append(check("total_population(district A)", total_population(north["children"][0]), 10))
results.append(check("total_population(North)", total_population(north), 30))
results.append(check("total_population(Toyland)", total_population(toyland), 60))
try:
    test_tree = build_tree(rows)
except Exception:
    test_tree = None
print()
if test_tree is None:
    print("  (build_tree isn't working yet, so the real-tree tests were skipped.)")
else:
    print("  On the real tree:")
    sindh_total = total_population(test_tree["children"][6])
    all_total = total_population(test_tree)
    results.append(check("round(total_population(Sindh))",
                         None if sindh_total is None else round(sindh_total), 50520994))
    results.append(check("round(total_population(tree))",
                         None if all_total is None else round(all_total), 231088952))
summary("Task 3", results)

  On the tiny tree:
  [PASS] total_population(district A) gave 10
  [PASS] total_population(North) gave 30
  [PASS] total_population(Toyland) gave 60

  On the real tree:
  [PASS] round(total_population(Sindh)) gave 50520994
  [PASS] round(total_population(tree)) gave 231088952

  Task 3: 5 of 5 tests passed.
  Well done! Move on to the next task.


In [26]:
banner("TASK 3 — Do the people add up?")

if ready:
    added_up = total_population(tree)
    if added_up is None:
        print("total_population isn't written yet.")
    else:
        print("Pakistan's own figure:    ", round(tree["population"]))
        print("Its districts, added up:  ", round(added_up))


########################################################################
#  TASK 3 — Do the people add up?
########################################################################
Pakistan's own figure:     231088952
Its districts, added up:   231088952


The districts add up exactly to Pakistan's own figure. (The decimals are there because the populations are estimates from a model, not a head count.)

## Section 5 — Do the people-exposed numbers add up too?

Every province also reports its own number of people exposed in each window. Does it match its districts added up?

**Given code:** total_exposed(place, k) adds up the people exposed in window k (0 = "01-31 Aug", ..., 8 = "14-20 Nov"). Same pattern as total_population.

In [27]:
banner("SECTION 5 — Khyber Pakhtunkhwa, window 15-21 Sep")

def total_exposed(place, k):
    """Return the people exposed in window k, added up over all districts."""
    if place["level"] == "district":
        return place["exposed"][k]
    total = 0
    for child in place["children"]:
        total = total + total_exposed(child, k)
    return total


if ready:
    print("Window", WINDOW_LABELS[4])
    print("  Khyber Pakhtunkhwa's own figure:", round(tree["children"][4]["exposed"][4]))
    print("  Its districts, added up:        ", round(total_exposed(tree["children"][4], 4)))


########################################################################
#  SECTION 5 — Khyber Pakhtunkhwa, window 15-21 Sep
########################################################################
Window 15-21 Sep
  Khyber Pakhtunkhwa's own figure: 313872
  Its districts, added up:         335001


They don't match! The next cell checks every province, in every window.

## Given code — audit(place, k)

audit(place, k) finds every place whose own number of people exposed in window k is more than 1% away from its districts added up. Each problem it finds is a list: [name, reported, added_up]. For example:

```
audit(tree, 4)  ->  [["Khyber Pakhtunkhwa", 313872, 335001]]
```

Read it: it's recursion again. A district has nothing to check (base case); any other place checks itself, then asks each child for its problems.

In [28]:
banner("SECTION 5 — Audit results, all 9 windows")

def audit(place, k):
    """Return a list of [name, reported, added_up] for every place inside this
    one whose reported people exposed (window k) differs from the sum of its
    districts by more than 1%."""
    if place["level"] == "district":          # BASE CASE
        return []

    problems = []                             # RECURSIVE CASE
    reported = place["exposed"][k]
    added_up = total_exposed(place, k)
    if reported > 0 and abs(reported - added_up) > 0.01 * reported:
        problems.append([place["name"], reported, added_up])
    for child in place["children"]:
        problems = problems + audit(child, k)
    return problems


if ready:
    for k in range(len(WINDOW_LABELS)):
        problems = audit(tree, k)
        if len(problems) == 0:
            print(WINDOW_LABELS[k], ": everything adds up")
        for p in problems:
            difference = round(p[2] - p[1])
            print(WINDOW_LABELS[k], ":", p[0], "reports", round(p[1]),
                  "but its districts add up to", round(p[2]), "(a difference of", difference, "people)")


########################################################################
#  SECTION 5 — Audit results, all 9 windows
########################################################################
01-31 Aug : everything adds up
25-31 Aug : everything adds up
01-07 Sep : everything adds up
08-14 Sep : Pakistan reports 19035056 but its districts add up to 18742538 (a difference of -292518 people)
15-21 Sep : Khyber Pakhtunkhwa reports 313872 but its districts add up to 335001 (a difference of 21129 people)
26 Sep-02 Oct : everything adds up
03-09 Oct : Khyber Pakhtunkhwa reports 223719 but its districts add up to 232437 (a difference of 8718 people)
11-17 Oct : everything adds up
14-20 Nov : everything adds up


**Question 2:** In two windows, Khyber Pakhtunkhwa's districts add up to tens of thousands more people than KP's own figure, from the same UN team and the same satellite images. Give two possible reasons.

**My answer:** : estimates or errors could mismatch

**Question 3:** The audit ignores differences under 1%. Why not flag every tiny difference?

**My answer:** : rounding or estimation, so a 1% threshold helps avoid unnecessary warnings.

## Section 6 — Which districts were hit hardest?

A district's peak is the most people exposed in any window. We rank the districts two ways:

- by number: people exposed at the peak
- by share: peak / population

**Given code:** all_districts(place) collects every district into one list. It's recursive too: can you spot the base case?

In [29]:
banner("SECTION 6 — The hardest-hit districts")

def all_districts(place):
    """Return a list of all district places inside this place."""
    if place["level"] == "district":
        return [place]
    found = []
    for child in place["children"]:
        found = found + all_districts(child)
    return found


if ready:
    by_number = []
    by_share = []
    for d in all_districts(tree):
        peak = max(d["exposed"])              # max(...) gives the biggest value in a list
        share = peak / d["population"]
        by_number.append([peak, d["name"]])
        by_share.append([share, d["name"]])

    by_number.sort(reverse=True)              # biggest first (sorts by the first item)
    by_share.sort(reverse=True)

    print("TOP 10 by NUMBER of people exposed at the peak:")
    for i in range(10):
        print("  ", i + 1, by_number[i][1], "-", round(by_number[i][0]), "people")

    print()
    print("TOP 10 by SHARE of the population exposed at the peak:")
    for i in range(10):
        print("  ", i + 1, by_share[i][1], "-", round(100 * by_share[i][0]), "%")


########################################################################
#  SECTION 6 — The hardest-hit districts
########################################################################
TOP 10 by NUMBER of people exposed at the peak:
   1 Sialkot - 1813308 people
   2 Khairpur - 1670439 people
   3 Kambar Shahdad Kot - 1403991 people
   4 Larkana - 1348798 people
   5 Badin - 1274371 people
   6 Dadu - 1262147 people
   7 Shikarpur - 1236438 people
   8 Jacobabad - 1151623 people
   9 Naushahro Feroze - 1138187 people
   10 Shaheed Benazir Abad - 1047050 people

TOP 10 by SHARE of the population exposed at the peak:
   1 Musakhel - 188 %
   2 Jacobabad - 94 %
   3 Jaffarabad - 93 %
   4 Kambar Shahdad Kot - 91 %
   5 Kashmore - 88 %
   6 Sohbatpur - 85 %
   7 Shikarpur - 84 %
   8 Larkana - 80 %
   9 Sujawal - 72 %
   10 Nasirabad - 72 %


**Question 4:** Which districts are in both lists? Sialkot is first by number but not in the top 10 by share. Why?

**My answer:** : Jacobabad, Kambar Shahdad Kot, Larkana and Shikarpur appear in both lists. Sialkot has a large exposed population but also a larger total population.

**Question 5:** If you were deciding which districts get help first, which ranking would you use, and why?

**My answer:**  : I would use the share ranking because it shows how much of a district's population is affected, while also considering the total number of people exposed.

## A data mystery — Musakhel

**Question 6:** The top district by share, Musakhel, has more than 100% of its people exposed. Run the cell below and look at its 9 numbers. What do you think happened, and what should we do about it?

**My answer:** : The estimates may contain an error or overlap, causing exposed people to exceed the population, so the data should be checked before making decisions.

In [57]:
banner("SECTION 6 — The Musakhel mystery")

if ready:
    for d in all_districts(tree):
        if d["name"] == "Musakhel":
            print("Musakhel population:", round(d["population"]))
            for k in range(len(WINDOW_LABELS)):
                print("  ", WINDOW_LABELS[k], ":", round(d["exposed"][k]), "people exposed")


########################################################################
#  SECTION 6 — The Musakhel mystery
########################################################################
Musakhel population: 175701
   01-31 Aug : 329598 people exposed
   25-31 Aug : 0 people exposed
   01-07 Sep : 5 people exposed
   08-14 Sep : 5 people exposed
   15-21 Sep : 4 people exposed
   26 Sep-02 Oct : 21 people exposed
   03-09 Oct : 13 people exposed
   11-17 Oct : 21 people exposed
   14-20 Nov : 44 people exposed


## Section 7 — Save your work for the lab

Given code: just run it. It walks your tree and saves one row per district (160 rows), with these columns:

```
province | district | population | peak_exposed | peak_share |
flood_km2 x 9 | exposed x 9
```

The province is now a column, so it can't get lost if the file is sorted. The lab uses the 9 windows to forecast when the water will be gone.

In [30]:
banner("SECTION 7 — Saving district_peak.csv")

OUT = "district_peak.csv"

if not ready:
    print("Task 1 (build_tree) isn't finished, so nothing was saved. The lab files")
    print("include a ready-made district_peak.csv.")
else:
    header = ["province", "district", "population", "peak_exposed", "peak_share"]
    for label in WINDOW_LABELS:
        header.append("flood_km2 " + label)
    for label in WINDOW_LABELS:
        header.append("exposed " + label)

    saved = 0
    with open(OUT, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for province in tree["children"]:
            for d in province["children"]:
                peak = max(d["exposed"])
                line = [province["name"], d["name"], d["population"], peak,
                        peak / d["population"]]
                line = line + d["flood"] + d["exposed"]
                writer.writerow(line)
                saved = saved + 1

    print("Saved", saved, "districts to", OUT, "  (expected 160)")
    print()
    print("The first 3 lines of your new file:")
    with open(OUT, "r", encoding="utf-8") as f:
        for i in range(3):
            print("  ", f.readline().strip()[:100], "...")


########################################################################
#  SECTION 7 — Saving district_peak.csv
########################################################################
Saved 160 districts to district_peak.csv   (expected 160)

The first 3 lines of your new file:
   province,district,population,peak_exposed,peak_share,flood_km2 01-31 Aug,flood_km2 25-31 Aug,flood_k ...
   Azad Kashmir,Bagh,378099.32103600004,329.965231,0.0008726945874853422,0.444474,0.0,0.271783,0.0,0.13 ...
   Azad Kashmir,Bhimber,380655.932043,4570.9765242329995,0.012008157865031377,10.846441,4.011444,2.2190 ...


Didn't work? No problem: the lab files include a ready-made district_peak.csv.

## Task 4 (bonus, if you finish early) — find_path(place, name)

**What to do**

Write find_path(place, name): the list of names from place down to the place called name, or None if it isn't there.

```
find_path(tree, "Dadu")      ->  ["Pakistan", "Sindh", "Dadu"]
find_path(tree, "Sindh")     ->  ["Pakistan", "Sindh"]
find_path(tree, "Atlantis")  ->  None
```

In [33]:
def find_path(place, name):
    pass

In [34]:
# TEST — Task 4: find_path      (given code: don't change this cell)
# Run the cell above first, and again every time you change find_path.
results = []
# A tiny made-up tree: Toyland -> North (A, B) and South (C)
def toy_district(name, population):
    return {"name": name, "level": "district", "population": population, "children": []}
north = {"name": "North", "level": "province", "population": 30,
         "children": [toy_district("A", 10), toy_district("B", 20)]}
south = {"name": "South", "level": "province", "population": 30,
         "children": [toy_district("C", 30)]}
toyland = {"name": "Toyland", "level": "country", "population": 60, "children": [north, south]}
print("  On the tiny tree:")
results.append(check('find_path(Toyland, "C")', find_path(toyland, "C"), ["Toyland", "South", "C"]))
results.append(check('find_path(Toyland, "North")', find_path(toyland, "North"), ["Toyland", "North"]))
results.append(check('find_path(Toyland, "Toyland")', find_path(toyland, "Toyland"), ["Toyland"]))
if find_path(toyland, "C") is not None:      # only once find_path gives back paths
    results.append(check('find_path(Toyland, "Z")', find_path(toyland, "Z"), None))
try:
    test_tree = build_tree(rows)
except Exception:
    test_tree = None
print()
if test_tree is None:
    print("  (build_tree isn't working yet, so the real-tree tests were skipped.)")
else:
    print("  On the real tree:")
    results.append(check('find_path(tree, "Dadu")', find_path(test_tree, "Dadu"), ["Pakistan", "Sindh", "Dadu"]))
    results.append(check('find_path(tree, "Bagh")', find_path(test_tree, "Bagh"), ["Pakistan", "Azad Kashmir", "Bagh"]))
    if find_path(test_tree, "Dadu") is not None:
        results.append(check('find_path(tree, "Atlantis")', find_path(test_tree, "Atlantis"), None))
summary("Task 4", results)

  On the tiny tree:
  [FAIL] find_path(Toyland, "C")
           gave     None
           expected ['Toyland', 'South', 'C']
           (None usually means the function isn't written yet,
            or it's missing a return)
  [FAIL] find_path(Toyland, "North")
           gave     None
           expected ['Toyland', 'North']
           (None usually means the function isn't written yet,
            or it's missing a return)
  [FAIL] find_path(Toyland, "Toyland")
           gave     None
           expected ['Toyland']
           (None usually means the function isn't written yet,
            or it's missing a return)

  On the real tree:
  [FAIL] find_path(tree, "Dadu")
           gave     None
           expected ['Pakistan', 'Sindh', 'Dadu']
           (None usually means the function isn't written yet,
            or it's missing a return)
  [FAIL] find_path(tree, "Bagh")
           gave     None
           expected ['Pakistan', 'Azad Kashmir', 'Bagh']
           (None usually mean

In [35]:
banner("TASK 4 (BONUS) — find_path")

if ready:
    print("find_path(tree, 'Dadu'):     ", find_path(tree, "Dadu"))
    print("find_path(tree, 'Islamabad'):", find_path(tree, "Islamabad"))


########################################################################
#  TASK 4 (BONUS) — find_path
########################################################################
find_path(tree, 'Dadu'):      None
find_path(tree, 'Islamabad'): None


**Question 7 (if you did Task 4):** What does find_path(tree, "Islamabad") return, the territory or the district? Why that one?

**My answer:** *(double-click to type)*

## Wrap-up — write one line for each

1. In your own words: what is a BASE CASE, and what is a RECURSIVE CASE?
2. Which ranking from Section 6 surprised you most?
3. One thing that still confuses you (bring it to the lab):

Next time, in the lab: did help reach the hardest-hit districts, and when will the water be gone?

In [ ]:
banner("END OF SESSION 2")